Titre projet info

Import des données

In [31]:
import pandas as pd
import matplotlib.pyplot as plt

# Import des données individuelles
url_individu = "https://object.data.gouv.fr/ministere-culture/FREQ_MUSEES/ENTREES_ET_CATEGORIES_DE_PUBLIC.csv"
df_individu = pd.read_csv(url_individu, sep=';')

print(df_individu.head())

  IDPatrimostat IDMuseofile                region departement dateappellation  \
0       0105301       M0963  Auvergne-Rhône-Alpes         Ain      01/02/2003   
1       0105301       M0963  Auvergne-Rhône-Alpes         Ain      01/02/2003   
2       0105301       M0963  Auvergne-Rhône-Alpes         Ain      01/02/2003   
3       0105301       M0963  Auvergne-Rhône-Alpes         Ain      01/02/2003   
4       0105301       M0963  Auvergne-Rhône-Alpes         Ain      01/02/2003   

  ferme anneefermeture   nom_du_musee  lien_avec            ville  \
0   NON            NaN  musée de Brou        NaN  BOURG EN BRESSE   
1   NON            NaN  musée de Brou        NaN  BOURG EN BRESSE   
2   NON            NaN  musée de Brou        NaN  BOURG EN BRESSE   
3   NON            NaN  musée de Brou        NaN  BOURG EN BRESSE   
4   NON            NaN  musée de Brou        NaN  BOURG EN BRESSE   

  codeInseeCommune  annee   payant  gratuit     total  individuel  scolaires  \
0            01053

Import de la seconde base de données

In [32]:
# Import des données totales
url = "https://static.data.gouv.fr/resources/frequentation-des-musees-de-france-1/20250827-121955/frequentation-totale-mdf-2001-a-2016-data-def9.xlsx"
df_totale = pd.read_excel(url, sheet_name=None)

# Pour voir les noms des feuilles disponibles
print(df_totale.keys())

# Pour accéder à une table précise
df_freq_totale = df_totale['FREQUENTATION TOTALE']
print(df_freq_totale.head())

df_freq_gratuite = df_totale['FREQ GRATUITE']
#print(df_freq_gratuite.head())

df_freq_payante = df_totale['FREQ PAYANTE']
#print(df_freq_payante.head())

dict_keys(['FREQUENTATION TOTALE', 'FREQ GRATUITE', 'FREQ PAYANTE'])
  REF DU MUSEE           NEW REGIONS  \
0      0105301  AUVERGNE-RHÔNE-ALPES   
1      0105302  AUVERGNE-RHÔNE-ALPES   
2      0106401  AUVERGNE-RHÔNE-ALPES   
3      0113701  AUVERGNE-RHÔNE-ALPES   
4      0119201  AUVERGNE-RHÔNE-ALPES   

                                      NOM DU MUSEE             VILLE  \
0                                    Musée du Brou   BOURG-EN-BRESSE   
1            Musée Départemental des Pays De l'Ain   BOURG-EN-BRESSE   
2  Musée de la Société d'Histoire et d'Archéologie            BRIORD   
3                 Musée Départemental du Revermont  TREFFORT-CUISIAT   
4                              Musée Archéologique          IZERNORE   

  Fréquentation   2001   2002   2003   2004   2005  ...   2008   2009   2010  \
0        Totale  74056  80303  70599  71456  68960  ...  60486  69815  74028   
1        Totale    NaN    NaN    NaN    NaN    NaN  ...    NaN    NaN    NaN   
2        Totale  

In [33]:
nb_musees = len(df_freq_totale)
nb_musees

1241

Dans la deuxième base, le tableau "payant" n'a pas les mêmes noms de variables que les deux autres.

In [34]:
df_freq_payante = df_freq_payante.rename(columns={"REF MUSEE": "REF DU MUSEE",})
df_freq_totale = df_freq_totale.rename(columns={"NEW REGIONS": "NOMREG",})
df_freq_gratuite = df_freq_gratuite.rename(columns={"NEW REGIONS": "NOMREG",})

Nous souhaitons transformer la deuxième base de données pour avoir une seule variable année.

In [35]:

# Colonnes identifiantes à conserver
id_vars = ["REF DU MUSEE", "NOMREG", "NOM DU MUSEE", "VILLE", "Fréquentation"]

# Colonnes années
annee_vars = [str(y) for y in range(2001, 2017)]

# Passage en format long
df_freq_totale = df_freq_totale.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

df_freq_gratuite = df_freq_gratuite.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

df_freq_payante = df_freq_payante.melt(
    id_vars = id_vars,
    value_vars = annee_vars,
    var_name = "annee",
    value_name = "frequentation"
)

print(df_freq_totale.head())
print(df_freq_gratuite.head())
print(df_freq_payante.head())

  REF DU MUSEE                NOMREG  \
0      0105301  AUVERGNE-RHÔNE-ALPES   
1      0105302  AUVERGNE-RHÔNE-ALPES   
2      0106401  AUVERGNE-RHÔNE-ALPES   
3      0113701  AUVERGNE-RHÔNE-ALPES   
4      0119201  AUVERGNE-RHÔNE-ALPES   

                                      NOM DU MUSEE             VILLE  \
0                                    Musée du Brou   BOURG-EN-BRESSE   
1            Musée Départemental des Pays De l'Ain   BOURG-EN-BRESSE   
2  Musée de la Société d'Histoire et d'Archéologie            BRIORD   
3                 Musée Départemental du Revermont  TREFFORT-CUISIAT   
4                              Musée Archéologique          IZERNORE   

  Fréquentation annee frequentation  
0        Totale  2001         74056  
1        Totale  2001           NaN  
2        Totale  2001           100  
3        Totale  2001         22234  
4        Totale  2001           204  
  REF DU MUSEE                NOMREG  \
0      0105301  AUVERGNE-RHÔNE-ALPES   
1      0105302  AU

Nous souhaitons connaître le nombre de musée n'ayant jamais communiqué leur fréquentation entre 2001 et 2016.

In [36]:
# Quelles valeurs, autres que numériques, peut prendre la fréquentation?
valeurs_non_numeriques = (
    df_freq_totale.loc[
        pd.to_numeric(df_freq_totale["frequentation"], errors="coerce").isna(),
        "frequentation"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(sorted(valeurs_non_numeriques))

['F', 'NC', "Retrait d'appelaltion", "Retrait d'appellation", 'SO', 'Transfert à Marseille - MUCEM', 'Transfert à Nice']


Combien de musées présentent ces modalités au moins un fois ?

In [37]:
freq = df_freq_totale["frequentation"]

# On garde les lignes où la valeur est NA ou n'est pas un nombre
ligne = freq.isna() | pd.to_numeric(freq, errors="coerce").isna()

# On crée une colonne avec les modalités
df_temp = df_freq_totale.loc[ligne, ["REF DU MUSEE", "frequentation"]].copy()

# On remplace les NA par le texte "NA"
df_temp["frequentation"] = df_temp["frequentation"].fillna("NA")

# On compte le nombre de musées différents pour chaque modalité
resultat = df_temp.groupby("frequentation")["REF DU MUSEE"].nunique()

print(resultat)

frequentation
F                                340
NA                               230
NC                               224
Retrait d'appelaltion              2
Retrait d'appellation              2
SO                                38
Transfert à Marseille - MUCEM      1
Transfert à Nice                   1
Name: REF DU MUSEE, dtype: int64


Nous souhaitons maintenant connaître le nombre de musées présentant,toutes les années, une de ces modalités non numériques.

In [38]:
# On marque les lignes où la fréquentation est NA ou non numérique
df_freq_totale["speciale"] = freq.isna() | pd.to_numeric(freq, errors="coerce").isna()

# Pour chaque musée, on regarde si toutes les lignes sont spéciales
resultat_toutes_annees = df_freq_totale.groupby("REF DU MUSEE")["speciale"].all()

# Nombre de musées concernés
nb_musees_toutes_annees = resultat_toutes_annees.sum()

pourcentage = (nb_musees_toutes_annees / nb_musees) * 100

print(f"{nb_musees_toutes_annees} musées n'ont aucune fréquentation renseignée entre 2001 et 2016, soit {pourcentage:.2f}%.")

79 musées n'ont aucune fréquentation renseignée entre 2001 et 2016, soit 6.37%.


On peut supprimer les lignes vides (non réponse totale)

In [39]:
# Suppression des lignes où toutes les colonnes sont vides
df_freq_totale = df_freq_totale.dropna(how='all')
df_freq_gratuite = df_freq_gratuite.dropna(how='all')
df_freq_payante = df_freq_payante.dropna(how='all')

On souhaite, pour la suite des analyses, créer deux colonnes : l'une avec les statuts du musée et l'autre avec seulement des variables numériques (on remplace par NaN les valeurs non numériques)

In [40]:
# Création de la colonne numérique seulement
df_freq_totale["freq_net"] = pd.to_numeric(df_freq_totale["frequentation"], errors="coerce")
df_freq_payante["freq_net"] = pd.to_numeric(df_freq_payante["frequentation"], errors="coerce")
df_freq_gratuite["freq_net"] = pd.to_numeric(df_freq_gratuite["frequentation"], errors="coerce")

# Correction d'une coquille (Retrait d'appelaltion))
df_freq_totale["frequentation"] = df_freq_totale["frequentation"].replace("Retrait d'appelaltion", "Retrait d'appellation")
df_freq_payante["frequentation"] = df_freq_payante["frequentation"].replace("Retrait d'appelaltion", "Retrait d'appellation")
df_freq_gratuite["frequentation"] = df_freq_gratuite["frequentation"].replace("Retrait d'appelaltion", "Retrait d'appellation")


# Fonction pour automatiser la création d'une colonne "Statut"
# Fonction mise à jour pour la création de la colonne "Statut"
def ajouter_colonne_statut_precis(df):
    """
    Création de la colonne Statut.
    """
    df_copy = df.copy()
    # Dictionnaire des correspondances pour les valeurs non numériques
    statut = {
        "F": "Fermé",
        "NA": "NA",
        "NC": "NA",
        "Retrait d'appelaltion": "Retrait d'appellation", # Corrige la coquille au passage
        "Retrait d'appellation": "Retrait d'appellation",
        "SO": "Sans Objet",
        "Transfert à Marseille - MUCEM": "Transfert",
        "Transfert à Nice": "Transfert"
    }
    
    # On applique le dictionnaire
    df_copy["Statut"] = df_copy["frequentation"].replace(statut)
    
    # On s'assure que les vraies valeurs vides (NaN) deviennent "NA"
    df_copy["Statut"] = df_copy["Statut"].fillna("NA")
    
    # Si la ligne a un chiffre valide dans freq_net, le statut devient "ouvert"
    df_copy.loc[df_copy["freq_net"].notna(), "Statut"] = "Ouvert"
    
    return df_copy

# Création de la nouvelle colonne
df_freq_totale = ajouter_colonne_statut_precis(df_freq_totale)
df_freq_payante = ajouter_colonne_statut_precis(df_freq_payante)
df_freq_gratuite = ajouter_colonne_statut_precis(df_freq_gratuite)

# Distribution des statuts
print(df_freq_totale["Statut"].value_counts(dropna=False)) # pas de Nan dans tous les cas

Statut
Ouvert                   15779
Fermé                     2249
NA                        1757
Sans Objet                  65
Retrait d'appellation        4
Transfert                    2
Name: count, dtype: int64


Nous devons aussi nettoyer la base de données df_individu

In [41]:
# Harmonisation des noms avec les bases de données de fréquentation totale
df_individu = df_individu.rename(columns={"region": "NOMREG", "IDPatrimostat": "REF DU MUSEE",})

# Selection des variables pertinentes pour l'étude
df_individu = df_individu[[
    "REF DU MUSEE", "NOMREG", "departement", 
    "ferme", "anneefermeture", "payant", "gratuit", 
    "total", "individuel", "scolaires" , "groupes_hors_scolaires", 
    'moins_18_ans_hors_scolaires', "_18_25_ans"]
    ]

# Résumé de toutes les colonnes
resume_complet = df_individu.describe(include='all')
print(resume_complet)


       REF DU MUSEE                NOMREG departement  ferme anneefermeture  \
count         12292                 12292       12292  12262           1190   
unique         1247                    19         102      3             41   
top         7218102  Auvergne-Rhône-Alpes       Paris    NON           2019   
freq             12                  1395         538  10982             68   
mean            NaN                   NaN         NaN    NaN            NaN   
std             NaN                   NaN         NaN    NaN            NaN   
min             NaN                   NaN         NaN    NaN            NaN   
25%             NaN                   NaN         NaN    NaN            NaN   
50%             NaN                   NaN         NaN    NaN            NaN   
75%             NaN                   NaN         NaN    NaN            NaN   
max             NaN                   NaN         NaN    NaN            NaN   

              payant       gratuit         total   

In [42]:
# Calcule le pourcentage de NA par colonne
pourcentage_na = (df_individu.isna().mean() * 100).round(2)

# Affiche le résultat
print(pourcentage_na)

REF DU MUSEE                    0.00
NOMREG                          0.00
departement                     0.00
ferme                           0.24
anneefermeture                 90.32
payant                          0.03
gratuit                         0.02
total                           0.02
individuel                     12.42
scolaires                      11.71
groupes_hors_scolaires         12.76
moins_18_ans_hors_scolaires    45.68
_18_25_ans                     57.68
dtype: float64


Beaucoup de non-réponses pour les colonnes "moins_18_ans_hors_scolaires" et "_18_25_ans", la non-réponse pour l'année de fermeture est différente (car les musées sont, pour la plupart, encore ouverts, ainsi, ils ne peuvent pas renseigner leurs années de fermeture)

Nous pouvons étudier la non-réponse totale, pour savoir s'il est pertinent de supprimer les lignes vides

In [43]:
# Liste des colonnes qui concernent la fréquentation
colonnes_freq = [
    'payant', 'gratuit', 'total', 'individuel', 'scolaires', 
    'groupes_hors_scolaires', 'moins_18_ans_hors_scolaires', '_18_25_ans'
]

# Filtre : True si toutes les colonnes_freq sont NA sur la ligne
filtre_non_reponse = df_individu[colonnes_freq].isna().all(axis=1)

# Lignes dans un nouveau tableau pour les étudier
df_non_reponse = df_individu[filtre_non_reponse]

# Totale
len(df_non_reponse) 

# 1 seule non réponse totale, on peut supprimer la ligne concernée
df_individu = df_individu.dropna(subset=colonnes_freq, how='all')